In [17]:
import glob
import os
import librosa
import numpy as np
import pandas as pd

In [18]:
ROOT_DIR = '../datasets/chords/'
SEGMENT_DURATION_SEC = 0.1 # 100ms
SEGMENT_FEATURES_CSV = 'segment_chroma_features.csv'
PITCH_CLASS_LABELS = ['c', 'c#', 'd', 'd#', 'e', 'f', 'f#', 'g', 'g#', 'a', 'a#', 'b']
CHORD_COLUMN_LABEL = 'chord'

# Extracting Chroma Features

In [19]:
def extract_segment_chroma(audio_path, segment_duration_sec):
    segment_data = []
    

Found 0 audio files for segment feature extraction.
No segment data extracted. CSV file not created.


In [10]:
import os
import glob
import librosa
import numpy as np
import pandas as pd

# --- Configuration ---
# Define the root directory containing your audio files
# Replace './chords' with the actual path if it's located elsewhere
ROOT_DIR = '../datasets/chords'

# Define the desired segment duration in seconds (e.g., 100ms)
SEGMENT_DURATION_SEC = 0.1

# Define the output CSV file for segment features
SEGMENT_FEATURES_CSV = 'segment_chroma_features.csv'

# Define the labels for the 12 pitch classes (C to B)
PITCH_CLASS_LABELS = ['c', 'c#', 'd', 'd#', 'e', 'f', 'f#', 'g', 'g#', 'a', 'a#', 'b']
# Define the name for the chord label column
CHORD_LABEL_COLUMN = 'chord'

# --- Feature Extraction Function ---

def extract_segment_chroma(audio_path, segment_duration_sec):
    """
    Loads audio, segments it, and extracts chroma features for each segment.
    Also extracts the note and chord type from the file path and name.

    Args:
        audio_path (str): Path to the audio file.
        segment_duration_sec (float): Duration of each segment in seconds.

    Returns:
        list: A list of dictionaries, where each dictionary contains
              features and metadata for one segment. Returns empty list on error.
    """
    segment_data = []
    try:
        y, sr = librosa.load(audio_path)

        # Calculate hop length in samples based on segment duration
        # This determines the start time of each segment/frame for feature extraction.
        hop_length_samples = int(segment_duration_sec * sr)

        # Ensure hop_length is at least 1
        if hop_length_samples < 1:
            hop_length_samples = 1
            print(f"Warning: Segment duration {segment_duration_sec}s too short for sample rate {sr}. Using hop_length=1.")

        # Standard n_fft value (FFT window size) - common values are 2048 or 4096
        # n_fft is typically larger than hop_length for overlapping frames and better frequency resolution.
        n_fft = 2048

        # Extract chroma features. librosa calculates features for frames
        # based on n_fft and hop_length. Each column in the output corresponds
        # to a frame starting at `frame_index * hop_length_samples`.
        chromagram = librosa.feature.chroma_stft(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length_samples)
        # The chromagram shape is (12, number_of_frames). Each column is a chroma vector for a frame.

        # --- Extract Note and Chord Type from File Path and Name ---
        path_parts = audio_path.split(os.sep)
        filename = os.path.basename(audio_path)
        file_base, file_ext = os.path.splitext(filename)

        note = 'unknown_note'
        chord_type = 'unknown_type'
        combined_label = 'unknown_chord'

        try:
            # Extract chord type from directory name (assuming .../chords/chord_type/...)
            chords_index = path_parts.index('chords')
            if chords_index + 1 < len(path_parts) - 1:
                 chord_type = path_parts[chords_index + 1]

            # Extract note from the filename (assuming format like Note-Octave-ChordType-Suffix.wav)
            filename_parts = file_base.split('-')
            if len(filename_parts) > 0:
                note = filename_parts[0] # Take the first part as the note

            # Combine note and chord type
            if note != 'unknown_note' and chord_type != 'unknown_type':
                # Simple concatenation, you might need more sophisticated logic
                # depending on your exact desired label format (e.g., 'Cmaj7', 'Dmin')
                # For now, let's just concatenate: Note + ChordType
                combined_label = note + chord_type
            elif note != 'unknown_note':
                 combined_label = note + '_unknown_type'
            elif chord_type != 'unknown_type':
                 combined_label = 'unknown_note_' + chord_type
            else:
                 # Keep 'unknown_chord' if neither could be extracted
                 pass

        except ValueError:
            # 'chords' directory not found in path
            print(f"Warning: 'chords' directory not found in path {audio_path}. Cannot extract label based on directory.")
            # Attempt to extract label components from filename if directory structure is unexpected
            try:
                 filename_parts = file_base.split('-')
                 if len(filename_parts) >= 3: # Assuming format Note-Octave-ChordType-...
                      note = filename_parts[0]
                      chord_type = filename_parts[2]
                      combined_label = note + chord_type
                 elif len(filename_parts) > 0:
                      # Just take the first part if structure is simpler
                      note = filename_parts[0]
                      combined_label = note + '_unknown_type'
                 else:
                      # Fallback to unknown
                      combined_label = 'unknown_chord'
            except Exception as file_parse_error:
                 print(f"Warning: Could not parse filename {filename} for label: {file_parse_error}")
                 combined_label = 'unknown_chord'


        # Iterate through each frame (column) of the chromagram
        n_frames = chromagram.shape[1]
        for frame_index in range(n_frames):
            # Get the chroma vector for the current frame
            chroma_frame = chromagram[:, frame_index]

            # Calculate the start time of the frame/segment in seconds
            frame_start_time = frame_index * segment_duration_sec

            # Prepare data for the row (one row per frame/segment)
            segment_row = {
                'filepath': audio_path,
                'start_time': frame_start_time,
                CHORD_LABEL_COLUMN: combined_label # Use the combined note+type label
            }
            # Add chroma features using the defined pitch class labels
            for i, feature in enumerate(chroma_frame):
                segment_row[PITCH_CLASS_LABELS[i]] = feature

            segment_data.append(segment_row)

    except Exception as e:
        print(f"Error processing {audio_path}: {e}")

    return segment_data

# --- Main Execution for CSV Creation ---

if __name__ == "__main__":
    all_audio_files = glob.glob(os.path.join(ROOT_DIR, '**', '*.wav'), recursive=True)
    print(f"Found {len(all_audio_files)} audio files for segment feature extraction.")

    all_segment_data = []

    for filepath in all_audio_files:
        segment_features = extract_segment_chroma(filepath, SEGMENT_DURATION_SEC)
        all_segment_data.extend(segment_features) # Add segments from this file to the main list

    if not all_segment_data:
        print("No segment data extracted. CSV file not created.")
    else:
        # Create a pandas DataFrame from the collected segment data
        df = pd.DataFrame(all_segment_data)

        # Define the desired column order for the CSV
        # filepath, start_time, then pitch class labels, then chord label
        csv_column_order = ['filepath', 'start_time'] + PITCH_CLASS_LABELS + [CHORD_LABEL_COLUMN]

        # Reorder columns if they exist in the DataFrame
        existing_columns_in_order = [col for col in csv_column_order if col in df.columns]
        df = df[existing_columns_in_order]

        # Save the DataFrame to a CSV file
        df.to_csv(SEGMENT_FEATURES_CSV, index=False)

        print(f"\nSuccessfully extracted segment features and saved to {SEGMENT_FEATURES_CSV}")
        print(f"CSV contains {len(all_segment_data)} segments.")



Found 3570 audio files for segment feature extraction.


/home/haikal-mpph/Documents/personal/tugas/kuliah/4ia01/skripsi/chord-recognition/venv/lib/python3.13/site-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(



Successfully extracted segment features and saved to segment_chroma_features.csv
CSV contains 110670 segments.
